# 10장 내보내고 띄우기: 어댑터 합치기, GGUF, Ollama·llama.cpp·vLLM

내 데이터로 LLM 파인튜닝하기 10장의 실습 노트북이에요. 웹 챕터: https://math-ai-matrix-pi.vercel.app/tracks/finetune/10-export-serve.html

- **이 노트북은 작성자가 돌려 보지 않았어요.** 출력 칸이 비어 있는 이유예요. 직접 돌려 나온 숫자를 `checks/ch10.csv`에 계산값과 나란히 적어 주세요.
- 흐름: 자원 확인 → 설치 → 설정 → 내 어댑터의 변화 크기 재기 → 합치기(두 가지) → 모델 카드·비공개 Hub → GGUF 만들기 → llama-server → Ollama → vLLM(여러 어댑터) → 기록
- 8장 노트북이 저장한 `ch08_lora` 폴더(어댑터)가 있어야 해요. Colab 세션이 바뀌었다면 Google Drive에서 다시 가져오세요.
- `PRESET = "1.5B"`가 기본이에요. 8장을 1.5B로 돌렸다면 그대로 끝까지 갈 수 있어요. 7B는 합칠 때 bf16 모델 15.23GB를 메모리에 올려야 하고, 디스크에는 합친 모델 15.23GB + bf16 GGUF 약 15.24GB + Q4_K_M 4.68GB가 함께 놓여요. 무료 Colab의 RAM·디스크 한도는 Google이 고정값으로 밝히지 않아서 [확인 필요]예요. 1절의 출력으로 먼저 확인하세요.
- 버전과 명령은 2026-10-11에 확인한 것이에요: PEFT 0.21.2, transformers 5.19.0, huggingface_hub 2.2.0, llama.cpp b11552, Ollama 0.40.2, vLLM 0.31.0.

In [ ]:
# 1. 자원 확인: GPU, 디스크, RAM
!nvidia-smi
!df -h /content
!free -g

## 2. 설치

합치기와 Hub 올리기에 쓰는 라이브러리만 버전을 고정해요. llama.cpp와 Ollama, vLLM은 각 절에서 따로 설치해요. vLLM은 다른 라이브러리 버전을 바꿀 수 있어서 맨 끝에 두었어요.

In [ ]:
%%capture
!pip install "peft==0.21.2" "transformers==5.19.0" "huggingface_hub==2.2.0" accelerate safetensors bitsandbytes

In [ ]:
import os, json, glob, math, time, subprocess
import torch, transformers, peft, huggingface_hub
print("peft", peft.__version__, "| transformers", transformers.__version__, "| huggingface_hub", huggingface_hub.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

## 3. 설정

`ADAPTER_DIR`는 8장에서 저장한 어댑터 폴더예요. `BASE_16`은 합칠 16비트 원본, `BASE_4BIT`는 학습 때 쓴 4비트판이에요. 계산값(`CALC`)은 실습 저장소 `results/ch10.json`에서 옮겼어요.

In [ ]:
PRESET = "1.5B"   # 8장을 7B로 돌렸다면 "7B"
PRESETS = {
    "7B":   {"BASE_16": "Qwen/Qwen2.5-7B-Instruct",   "BASE_4BIT": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"},
    "1.5B": {"BASE_16": "Qwen/Qwen2.5-1.5B-Instruct", "BASE_4BIT": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"},
}
BASE_16, BASE_4BIT = PRESETS[PRESET]["BASE_16"], PRESETS[PRESET]["BASE_4BIT"]
ADAPTER_DIR = "ch08_lora"            # 예: "/content/drive/MyDrive/ch08_lora"
HF_REPO = "your-id/ours-qwen2.5-tone-lora"   # 비공개로 올릴 저장소 이름
PUSH = False                         # True로 바꾸면 Hub에 올려요(토큰 필요)
CTX = 4096
CALC = {"7B": {"merged_gb": 15.23, "gguf_bf16_gb": 15.238, "gguf_q4km_gb": 4.683, "kv_per_token": 57344,
               "tok_s_upper_L4_bf16": 19.7, "tok_s_upper_T4_q4km": 68.33}}
VAL_PROMPTS = ["주문한 지 사흘이 지났는데 아직 배송 출발을 안 했어요. 언제 오나요?",
               "사이즈가 작아서 교환하고 싶어요. 배송비가 드나요?"]
SYSTEM = "당신은 우리 쇼핑몰 고객 문의에 답변 초안을 쓰는 도우미예요."
def gb(path):
    return sum(os.path.getsize(p) for p in glob.glob(path, recursive=True) if os.path.isfile(p)) / 1e9

## 4. 내 어댑터가 더한 변화는 원래 가중치의 몇 배일까

웹 챕터의 합치기 장난감은 변화의 크기 `rho`(sBA의 크기 ÷ W의 크기)를 가정값으로 훑었어요. 여기서 내 어댑터의 실제 값을 재서 실습 저장소 `data/scenario.json`의 `rho_main`에 넣어 보세요. 첫 층 `q_proj` 하나만 재요.

`adapter_config.json`의 `base_model_name_or_path`도 함께 찍어요. 4비트판으로 학습했다면 여기에 4비트 저장소 이름이 적혀 있을 수 있어요. 16비트 원본에 붙일 때는 이 값을 믿지 말고 기본 모델을 직접 적어 주세요.

In [ ]:
from safetensors.torch import load_file
from safetensors import safe_open
from huggingface_hub import hf_hub_download

cfg = json.load(open(f"{ADAPTER_DIR}/adapter_config.json"))
print("base_model_name_or_path:", cfg.get("base_model_name_or_path"), "| r:", cfg["r"], "| lora_alpha:", cfg["lora_alpha"])
s = cfg["lora_alpha"] / cfg["r"]
ad = load_file(f"{ADAPTER_DIR}/adapter_model.safetensors")
kA = next(k for k in ad if "layers.0.self_attn.q_proj.lora_A" in k)
kB = next(k for k in ad if "layers.0.self_attn.q_proj.lora_B" in k)
delta = s * (ad[kB].float() @ ad[kA].float())

name = "model.layers.0.self_attn.q_proj.weight"
try:   # 7B는 여러 조각이라 목차 파일이 있고, 1.5B는 model.safetensors 한 파일이에요
    shard = json.load(open(hf_hub_download(BASE_16, "model.safetensors.index.json")))["weight_map"][name]
except Exception:
    shard = "model.safetensors"
with safe_open(hf_hub_download(BASE_16, shard), "pt") as f:
    W = f.get_tensor(name).float()
rms = lambda t: t.pow(2).mean().sqrt().item()
print("adapter dtype:", ad[kA].dtype, "| rho = rms(sBA)/rms(W) =", round(rms(delta) / rms(W), 5))

## 5. 합치기: 두 가지 바탕

- **C. 16비트 원본에 합치기**: `BASE_16`을 bf16으로 불러 어댑터를 얹고 `merge_and_unload()`. 학습 때 본 가중치(4비트를 되돌린 값)와 바탕이 조금 달라요.
- **B. 4비트를 되돌린 가중치에 합치기**: `BASE_4BIT`를 불러 `dequantize()`로 16비트로 되돌린 뒤 합쳐요. 학습 때와 같은 바탕이지만, 4비트 양자화 오차가 그대로 남은 모델이에요.

PEFT 0.21.2는 4비트 층에 그대로 합치면 "Merge lora module to 4-bit linear may get different generations due to rounding errors." 경고를 내고, 합친 값을 다시 4비트로 담아요. 이 노트북은 그 길을 쓰지 않아요.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(BASE_16)
base = AutoModelForCausalLM.from_pretrained(BASE_16, dtype=torch.bfloat16, low_cpu_mem_usage=True)
merged_c = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()   # 제자리 변경이 아니라 돌려받은 모델을 써요
merged_c.save_pretrained("merged_c", max_shard_size="5GB")
tok.save_pretrained("merged_c")
print("merged_c safetensors", round(gb("merged_c/*.safetensors"), 3), "GB | 7B 계산값", CALC["7B"]["merged_gb"], "GB")
del base, merged_c

In [ ]:
DO_PATH_B = PRESET == "1.5B"   # 7B는 GPU 메모리가 빠듯해서 기본으로 끔
if DO_PATH_B:
    q = AutoModelForCausalLM.from_pretrained(BASE_4BIT, device_map="cuda")
    q.dequantize()   # 4비트 층을 16비트로 되돌려요
    merged_b = PeftModel.from_pretrained(q, ADAPTER_DIR).merge_and_unload()
    merged_b.save_pretrained("merged_b", max_shard_size="5GB"); tok.save_pretrained("merged_b")
    print("merged_b", round(gb("merged_b/*.safetensors"), 3), "GB")
    del q, merged_b
    torch.cuda.empty_cache()

In [ ]:
# 학습 때와 같은 조건(4비트 + 어댑터 따로)과 합친 모델의 답을 나란히 봐요. 탐욕적 생성(do_sample=False)이라 매번 같아요.
def answer(model, text):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]
    # transformers 5.19.0은 return_dict 기본값이 True라서 텐서만 받으려면 False로 적어요
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=False).to(model.device)
    out = model.generate(ids, max_new_tokens=128, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

runs = {"A 4비트+어댑터": lambda: PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(BASE_4BIT, device_map="cuda"), ADAPTER_DIR)}
if PRESET == "1.5B":   # 7B 합친 모델(fp16 15.23GB)은 T4 16GB에 올리지 않아요. 더 큰 GPU라면 이 조건을 풀어요
    runs["C 원본에 합침"] = lambda: AutoModelForCausalLM.from_pretrained("merged_c", dtype=torch.float16, device_map="cuda")
if DO_PATH_B:
    runs["B 되돌려 합침"] = lambda: AutoModelForCausalLM.from_pretrained("merged_b", dtype=torch.float16, device_map="cuda")
for label, load in runs.items():
    m = load()
    for p in VAL_PROMPTS:
        print(f"[{label}] {p}\n -> {answer(m, p)}\n")
    del m; torch.cuda.empty_cache()

## 6. 모델 카드와 비공개 Hub 저장소

모델 카드 머리말에 기본 모델(`base_model`), 라이선스(`license`), 관계(`base_model_relation`)를 적고, 데이터 출처는 본문에 글로 적어요. Hub에 올린 데이터셋이 아니면 `datasets` 칸은 비워 둬요.

`create_repo(..., private=True)`는 저장소가 이미 있으면 무시돼요(huggingface_hub 2.2.0 문서). 그래서 올린 뒤 `repo_info(...).private`로 정말 비공개인지 확인해요.

In [ ]:
CARD = f'''---
license: apache-2.0
base_model: {BASE_16}
base_model_relation: adapter
library_name: peft
language:
- ko
---

# 문의 답변 초안 어댑터

- 기본 모델: {BASE_16} (Apache-2.0). 학습은 4비트판 {BASE_4BIT} 위에서 했어요.
- 학습 방법: QLoRA, 모든 선형층, r = {cfg["r"]}, lora_alpha = {cfg["lora_alpha"]} (8장 설정).
- 데이터 출처: 우리 팀이 쓴 지난 문의·답변(비공개). 개인정보는 학습 전에 지웠어요.
- 쓰면 안 되는 곳: 바뀌는 정책(환불 기한, 배송비)은 학습한 값을 믿지 말고 검색으로 붙여요.
'''
open(f"{ADAPTER_DIR}/README.md", "w").write(CARD)
if PUSH:
    from huggingface_hub import create_repo, upload_folder, repo_info
    create_repo(HF_REPO, private=True, exist_ok=True)
    upload_folder(repo_id=HF_REPO, folder_path=ADAPTER_DIR)
    print("private:", repo_info(HF_REPO).private)

## 7. GGUF 만들기 (llama.cpp b11552)

합친 모델을 `convert_hf_to_gguf.py`로 bf16 GGUF로 바꾸고, `llama-quantize`로 Q4_K_M을 만들어요. Ollama 0.34.1부터는 safetensors를 GGUF로 바꾸거나 양자화하는 일을 Ollama가 하지 않아서, 이 단계를 llama.cpp로 해요.

주의: llama.cpp의 `requirements.txt`는 transformers 4를 설치해요(quantize README). 그래서 앞 절에서 고정한 transformers 5.19.0이 바뀌어요. 합치기와 답 비교를 이 절보다 먼저 끝내 두고, 앞 절을 다시 돌리려면 런타임을 다시 시작하세요.

In [ ]:
%%capture
!git clone --depth 1 --branch b11552 https://github.com/ggml-org/llama.cpp
!pip install -r llama.cpp/requirements.txt
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON
!cmake --build llama.cpp/build --config Release -j 2 --target llama-quantize llama-server

In [ ]:
!python llama.cpp/convert_hf_to_gguf.py merged_c --outtype bf16 --outfile ours-bf16.gguf
!./llama.cpp/build/bin/llama-quantize ours-bf16.gguf ours-Q4_K_M.gguf Q4_K_M
print("bf16 GGUF", round(gb("ours-bf16.gguf"), 3), "GB | Q4_K_M", round(gb("ours-Q4_K_M.gguf"), 3), "GB")
print("7B 계산값: bf16", CALC["7B"]["gguf_bf16_gb"], "GB, Q4_K_M", CALC["7B"]["gguf_q4km_gb"], "GB")

## 8. llama-server로 띄우기

`llama-server`는 OpenAI와 같은 모양의 `/v1/chat/completions`를 열어요. b11552의 기본 포트는 9931이고, 문맥 길이 `-c`를 적지 않으면 모델에 적힌 값을 써요. 응답의 `timings.predicted_per_second`가 생성 속도예요.

어댑터를 합치지 않고 따로 두고 싶다면 `convert_lora_to_gguf.py`로 어댑터를 GGUF로 바꿔 `--lora`로 붙일 수 있어요(아래 두 번째 칸). 이때 바탕은 16비트 원본으로 만든 GGUF여야 해요.

In [ ]:
import requests
srv = subprocess.Popen(["./llama.cpp/build/bin/llama-server", "-m", "ours-Q4_K_M.gguf", "-c", str(CTX), "--port", "9931", "-ngl", "99"],
                       stdout=open("llama_server.log", "w"), stderr=subprocess.STDOUT)
time.sleep(30)
r = requests.post("http://127.0.0.1:9931/v1/chat/completions", json={
    "messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": VAL_PROMPTS[0]}],
    "max_tokens": 200, "temperature": 0}).json()
print(r["choices"][0]["message"]["content"])
print("생성 속도(토큰/초):", r.get("timings", {}).get("predicted_per_second"), "| 7B T4 계산상 상한", CALC["7B"]["tok_s_upper_T4_q4km"])
srv.terminate()

In [ ]:
# (선택) 어댑터를 따로 두는 길: 원본 GGUF + 어댑터 GGUF
!python llama.cpp/convert_hf_to_gguf.py {BASE_16.split('/')[-1]} --outtype bf16 --outfile base-bf16.gguf 2>/dev/null || echo "원본 폴더가 없으면 huggingface_hub.snapshot_download로 먼저 받으세요"
!python llama.cpp/convert_lora_to_gguf.py {ADAPTER_DIR} --base-model-id {BASE_16} --outtype bf16 --outfile ours-lora-bf16.gguf
# !./llama.cpp/build/bin/llama-server -m base-bf16.gguf --lora ours-lora-bf16.gguf -c 4096 --port 9931

## 9. Ollama로 띄우기 (0.40.2)

Modelfile의 `FROM`에 GGUF 파일을 적어요. Qwen2.5의 대화 틀(ChatML)은 `TEMPLATE`에 직접 적고, `num_ctx`도 직접 적어요. Modelfile 문서는 기본값을 2048로, FAQ는 4096으로 적고 있어서 적지 않으면 어느 쪽인지 헷갈려요.

0.34.1부터 Ollama는 LoRA 어댑터를 받지 않아요. 두 번째 칸은 `ADAPTER` 줄을 넣으면 어떤 오류가 나는지 확인하는 칸이에요.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
ollama_srv = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(10)
open("Modelfile", "w").write(f'''FROM ./ours-Q4_K_M.gguf
TEMPLATE """{{{{ if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}{{{{ if .Prompt }}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
"""
SYSTEM """{SYSTEM}"""
PARAMETER num_ctx {CTX}
PARAMETER temperature 0.3
PARAMETER stop "<|im_end|>"
''')
!ollama create ours-tone -f Modelfile
!ollama run ours-tone "{VAL_PROMPTS[1]}"
!ollama --version

In [ ]:
# 확인용: ADAPTER 줄이 든 Modelfile은 ollama create에서 거절돼야 해요("LoRA adapters are no longer supported").
open("Modelfile.adapter", "w").write(f"FROM ours-tone\nADAPTER ./ours-lora-bf16.gguf\n")
!ollama create ours-tone-adapter -f Modelfile.adapter

## 10. vLLM으로 여러 어댑터 띄우기 (0.31.0)

`--enable-lora`로 기본 모델 하나에 어댑터를 이름 붙여 올리고, 요청의 `model` 칸에 어댑터 이름을 적어요. vLLM 0.31.0 기본값은 `max_lora_rank` 16, `max_loras` 1(한 묶음에 섞을 수 있는 어댑터 수), `gpu_memory_utilization` 0.92예요.

T4는 bf16을 지원하지 않아서 `--dtype half`로 적어요. 7B bf16(15.23GB)은 16GB × 0.92 = 14.72GB에 들어가지 않으니 T4에서는 1.5B로 해요. vLLM 설치가 앞의 라이브러리 버전을 바꿀 수 있어서, 새 런타임에서 이 절만 돌리는 편이 안전해요.

In [ ]:
%%capture
!pip install "vllm==0.31.0" openai

In [ ]:
vsrv = subprocess.Popen(["vllm", "serve", BASE_16, "--enable-lora",
                         "--lora-modules", f"tone={ADAPTER_DIR}", f"tone2={ADAPTER_DIR}",
                         "--max-lora-rank", "16", "--max-loras", "2", "--dtype", "half", "--max-model-len", "2048"],
                        stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
time.sleep(240)
!curl -s localhost:8000/v1/models

In [ ]:
# 요청 1개와 8개를 동시에 보내 합계 토큰/초를 재요. 어댑터 두 이름을 섞어 보내요.
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
def one(i):
    r = client.chat.completions.create(model=["tone", "tone2"][i % 2], max_tokens=200, temperature=0,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": VAL_PROMPTS[i % 2]}])
    return r.usage.completion_tokens
for n in (1, 8):
    t0 = time.time()
    with ThreadPoolExecutor(n) as ex:
        toks = sum(ex.map(one, range(n)))
    print(f"동시 {n}개: {toks} 토큰 / {time.time() - t0:.1f}초 = {toks / (time.time() - t0):.1f} 토큰/초")
vsrv.terminate()

In [ ]:
# 11. checks/ch10.csv에 붙여 넣을 한 줄(빈칸은 위 출력에서 옮겨 적어요)
import datetime
print(",".join(str(x) for x in [datetime.date.today().isoformat(), torch.cuda.get_device_name(0), PRESET, BASE_16,
      round(gb("merged_c/*.safetensors"), 3), round(gb("ours-bf16.gguf"), 3), round(gb("ours-Q4_K_M.gguf"), 3),
      "", "", "", "", f"peft {peft.__version__} / transformers {transformers.__version__}"]))